### Final comparison: MF vs NCF

Only MF and NCF are final experiment targets. GMF and MLP are internal pretraining stages.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


In [ ]:
from src.models import MatrixFactorization, NCF
from src.training import evaluate_candidates

validation_candidates = pd.read_csv(PROCESSED / "validation_candidates_100neg.csv")
test_candidates = pd.read_csv(PROCESSED / "test_candidates_100neg.csv")
mf_checkpoint = torch.load(MODELS / "mf_best.pth", map_location=DEVICE)
ncf_checkpoint = torch.load(MODELS / "ncf_best.pth", map_location=DEVICE)
mf = MatrixFactorization(mf_checkpoint["num_users"], mf_checkpoint["num_items"], embedding_dim=mf_checkpoint["config"]["embedding_dim"]).to(DEVICE)
mf.load_state_dict(mf_checkpoint["model_state_dict"]); mf.eval()
ncf = NCF(ncf_checkpoint["num_users"], ncf_checkpoint["num_items"], predictive_factor=ncf_checkpoint["config"]["predictive_factor"]).to(DEVICE)
ncf.load_state_dict(ncf_checkpoint["model_state_dict"]); ncf.eval()


In [ ]:
rows = []
for model_name, model in (("MF", mf), ("NCF", ncf)):
    for evaluation, candidates in (("validation", validation_candidates), ("test", test_candidates)):
        rows.append({"model": model_name, "evaluation": evaluation, **evaluate_candidates(model, candidates, DEVICE)})
metrics = pd.DataFrame(rows)
metrics


In [ ]:
metrics[metrics.model == "MF"].to_csv(RESULTS / "mf_metrics.csv", index=False)
metrics[metrics.model == "NCF"].to_csv(RESULTS / "ncf_metrics.csv", index=False)
metrics[metrics.evaluation == "test"].to_csv(RESULTS / "final_mf_vs_ncf_comparison.csv", index=False)
print("saved MF vs NCF comparison")